In [21]:
%cd Desktop/Harris-2026/data-2/final-project-cameron-and-yvaine

import geopandas as gpd
import os

in_path = "data/raw-data/cb_2024_us_county_500k"
county_geo = os.path.join(in_path, "cb_2024_us_county_500k.shp")

counties = gpd.read_file(county_geo)


# 1. Only keep the four columns we need
counties = counties[['GEOID', 'NAMELSAD', 'STUSPS', 'STATE_NAME', 'geometry']]

# 2. Rename the STUSPS column to state, and also rename the other columns for clarity
counties = counties.rename(columns={'GEOID':'geoid','STUSPS': 'state','NAMELSAD':'county_name','STATE_NAME':'state_name'})
# 3. Check the first few rows of the cleaned data
print(counties.head())

# 4. Define the output file path
derived_path = "data/derived-data"
out_file = os.path.join(derived_path, "counties.geojson")

# 5. Save the cleaned GeoDataFrame to a new file
# driver='GeoJSON'
counties.to_file(out_file, driver='GeoJSON')

# 检查唯一性
dup = (
    counties.groupby(["county_name", "state_name"])
    .size().reset_index(name="n").query("n>1")
)
print("duplicated pairs (should be 0):", len(dup))


[Errno 2] No such file or directory: 'Desktop/Harris-2026/data-2/final-project-cameron-and-yvaine'
/Users/rose./Desktop/Harris-2026/data-2/final-project-cameron-and-yvaine
   geoid     county_name state state_name  \
0  01069  Houston County    AL    Alabama   
1  01023  Choctaw County    AL    Alabama   
2  01113  Russell County    AL    Alabama   
3  10005   Sussex County    DE   Delaware   
4  01071  Jackson County    AL    Alabama   

                                            geometry  
0  POLYGON ((-85.71209 31.19727, -85.70934 31.198...  
1  POLYGON ((-88.47323 31.89386, -88.46888 31.930...  
2  POLYGON ((-85.4347 32.31761, -85.43384 32.3922...  
3  POLYGON ((-75.7226 38.82986, -75.61542 38.8336...  
4  MULTIPOLYGON (((-86.15423 34.53378, -86.14989 ...  
duplicated pairs (should be 0): 0


In [ ]:
f = pd.read_csv("data/derived/mobility_rate.csv")

# 暂时造一个 fake EAS
import numpy as np
df["eas"] = np.random.normal(0.5, 0.1, len(df))

chart = (
    alt.Chart(df)
    .mark_circle(size=60)
    .encode(
        x=alt.X("eas:Q", title="Education Access Score"),
        y=alt.Y("mobility_rate:Q", title="Mobility Rate"),
        tooltip=["county_name", "state", "eas", "mobility_rate"]
    )
    .properties(
        title="Education Access vs Intergenerational Mobility",
        width=600,
        height=400
    )
)
+ alt.Chart(df).mark_line(color="red").transform_regression("eas","mobility_rate")



In [13]:
in_path = "data/raw-data"
county_geo = os.path.join(in_path, "cb_2024_us_county_500k","cb_2024_us_county_500k.shp")

counties = gpd.read_file(county_geo)

print(counties.head(5))

# 1. Only keep the four columns we need
counties = counties[['GEOID','NAME', 'STUSPS', 'STATE_NAME', 'geometry']]
# 2. Rename the STUSPS column to state, and also rename the other columns for clarity
counties = counties.rename(columns={'GEOID':'geoid','STUSPS': 'state','NAME':'county_name','STATE_NAME':'state_name'})
# 3. Check the first few rows of the cleaned data
print(counties.head())

# 6. geoid for furture calculation and merging
#since conty name is not unique across states, we will create a new column that combines county_name and state to ensure uniqueness for future merging
counties_keys = counties[["geoid", "county_name", "state"]].copy()
counties_keys["geoid"] = counties_keys["geoid"].astype(str)


dup = (
    counties_keys
    .groupby(["county_name", "state"])
    .size()
    .reset_index(name="n")
    .query("n > 1")
)
print(dup.head(20))
print("duplicated pairs:", len(dup))

  STATEFP COUNTYFP  COUNTYNS         GEOIDFQ  GEOID     NAME        NAMELSAD  \
0      01      069  00161560  0500000US01069  01069  Houston  Houston County   
1      01      023  00161537  0500000US01023  01023  Choctaw  Choctaw County   
2      01      113  00161583  0500000US01113  01113  Russell  Russell County   
3      10      005  00217269  0500000US10005  10005   Sussex   Sussex County   
4      01      071  00161561  0500000US01071  01071  Jackson  Jackson County   

  STUSPS STATE_NAME LSAD       ALAND     AWATER  \
0     AL    Alabama   06  1501742250    4795418   
1     AL    Alabama   06  2365900084   19114321   
2     AL    Alabama   06  1660653961   15562947   
3     DE   Delaware   06  2424590442  674129051   
4     AL    Alabama   06  2792044612  126334711   

                                            geometry  
0  POLYGON ((-85.71209 31.19727, -85.70934 31.198...  
1  POLYGON ((-88.47323 31.89386, -88.46888 31.930...  
2  POLYGON ((-85.4347 32.31761, -85.43384 32.39

In [ ]:
#================================
# Calculating EAS
#================================

from shapely.geometry import Point
import pyproj
import matplotlib.pyplot as plt

# drop counties in U.S. territories
counties = counties[~counties['state'].isin(['AK', 'HI', 'AS', 'GU', 'MP', 'PR', 'VI'])]


MILES_TO_METERS = 1609.344
RADII_MILES = [25, 50, 75]
PROJECTED_CRS = "EPSG:5070"   # meters

# -------------------------
# 1) Read counties (polygons)
# -------------------------
derived_path = "data/derived-data"
counties = gpd.read_file(os.path.join(derived_path, "counties.geojson"))

# Ensure a known CRS (most GeoJSON are EPSG:4326)
if counties.crs is None:
    counties = counties.set_crs("EPSG:4326")

# -------------------------
# 2) Read schools (points)
# -------------------------
schools = pd.read_csv(os.path.join(derived_path, "schools_gdf.csv"))  

# -------------------------
# 3) Project both to meters
# -------------------------
counties_p = counties.to_crs(PROJECTED_CRS)
schools_p = schools.to_crs(PROJECTED_CRS)

# -------------------------
# 4) County "centroid" points (use representative_point for safety)
# -------------------------
county_pts = counties_p[["GEOID"]].copy()
county_pts["geometry"] = counties_p.geometry.representative_point()
county_pts = gpd.GeoDataFrame(county_pts, geometry="geometry", crs=PROJECTED_CRS)

# -------------------------
# 5) Buffer rings and count schools within each radius
# -------------------------
out = counties_p[["GEOID"]].copy()

for r in RADII_MILES:
    buf = county_pts.copy()
    buf["geometry"] = buf.geometry.buffer(r * MILES_TO_METERS)
    buf = gpd.GeoDataFrame(buf, geometry="geometry", crs=PROJECTED_CRS)

    # spatial join: each school matched to the county buffer it falls within
    joined = gpd.sjoin(
        schools_p[["geometry"]],
        buf[["GEOID", "geometry"]],
        how="inner",
        predicate="within"
    )

    counts = joined.groupby("GEOID").size().rename(f"schools_within_{r}mi")
    out = out.merge(counts, on="GEOID", how="left")

# fill missing with 0
for r in RADII_MILES:
    col = f"schools_within_{r}mi"
    out[col] = out[col].fillna(0).astype(int)

# -------------------------
# 6) Save derived data (numerator components)
# -------------------------
out.to_csv("data/derived-data/county_school_counts_by_radius.csv", index=False)
out.head()